In [2]:
import pandas as pd
import numpy as np
import os 
import glob
import geopandas as gpd
import ee
from shapely.geometry import point

In [3]:
#initialize GEE
ee.Initialize(project='western-octagon-493416-g5')

#Paths

Transmission_Path = r"D:\Profession\My_Space\DS\ML\My_Projects\Vegetation_Encroachment_Risk_Detection_California_Transmission_Lines\Data\transmission_lines"
Fire_Path = r"D:\Profession\My_Space\DS\ML\My_Projects\Vegetation_Encroachment_Risk_Detection_California_Transmission_Lines\Data\fire_incidents"
Output_path=r"D:\Profession\My_Space\DS\ML\My_Projects\Vegetation_Encroachment_Risk_Detection_California_Transmission_Lines\outputs"


In [4]:
lines=gpd.read_file(os.path.join(Transmission_Path,'TransmissionLine_CEC.shp'))
lines.head()

,Name,kV,kV_Sort,Owner,Status,Circuit,Type,Legend,Length_Mil,Length_Fee,TLine_Name,Source,Comments,Creator,Creator_Da,Last_Edito,Last_Edi_1,GlobalID,geometry
0,AMP 115kV,115,115.0,AMP,Operational,Single,OH,Other_110_161kV,2.0,10879.98077483,NaN,AMP,NaN,FTHONG,2012-08-31,SVC_AGIS_SQLADM,2016-04-25,e250d3dd-0564-42f8-822f-73f741c17218,"LINESTRING (-13608018.951 4547326.915, -136080..."
1,AMP 115kV,115,115.0,AMP,Operational,Single,OH,Other_110_161kV,3.0,16505.41123852,NaN,AMP,NaN,FTHONG,2012-08-31,SVC_AGIS_SQLADM,2016-04-25,16eb0e69-4e5b-4af0-acce-37915f94d9b3,"LINESTRING (-13608044.414 4547272.768, -136080..."
2,AMP 115kV,115,115.0,AMP,Operational,Single,OH,Other_110_161kV,1.0,3807.24014248,NaN,AMP,NaN,FTHONG,2012-08-31,SVC_AGIS_SQLADM,2016-04-25,c1c874a0-f619-4537-8537-3f4cdb4e0982,"LINESTRING (-13613422.397 4548276.164, -136134..."
3,AMP 115kV,115,115.0,AMP,Operational,Single,OH,Other_110_161kV,1.0,6135.61557739,NaN,AMP,Partially underwater,FTHONG,2012-08-31,SVC_AGIS_SQLADM,2016-04-25,7809d1a4-1fb9-4841-bf9a-192acb047266,"LINESTRING (-13613131.635 4549619.954, -136131..."
4,ANZA 34kV,34,34.0,ANZA,Operational,Single,OH,Other_33_92kV,24.0,127189.40561552,NaN,City of Anza,NaN,FTHONG,2012-08-24,SVC_AGIS_SQLADM,2016-04-25,e26bc920-7d3b-404d-9b5e-0d65db992b62,"LINESTRING (-12994452.454 3989231.906, -129944..."


In [5]:
# Load transmission lines
lines = gpd.read_file(os.path.join(Transmission_Path, "TransmissionLine_CEC.shp"))

# Convert kV to numeric
lines['kV'] = pd.to_numeric(lines['kV'], errors='coerce')

# Filter - overhead and operational only
lines_filtered = lines[
    (lines['Status'] == 'Operational') &
    (lines['Type'] == 'OH')
]

print(f"Total lines: {len(lines)}")
print(f"Filtered lines: {len(lines_filtered)}")
print(f"CRS: {lines_filtered.crs}")

Total lines: 6839
Filtered lines: 6675
CRS: EPSG:3857


In [6]:
# Load all fire incident files
dfs = []

for file in glob.glob(os.path.join(Fire_Path, "*.xlsx")):
    df = pd.read_excel(file, header=1)
    dfs.append(df)

# Merge all
fire_df = pd.concat(dfs, ignore_index=True)

# Clean column names
fire_df.columns = fire_df.columns.str.strip().str.replace('\n', ' ')

print(f"Total fire incidents: {len(fire_df)}")
print(f"Columns: {list(fire_df.columns)}")

Total fire incidents: 1786
Columns: ['Unnamed: 0', 'Unnamed: 1', 'Date', 'Time', 'Latitude', 'Longitude', 'Material at Origin', 'Material at Origin - Comments', 'Land Use at Origin', 'Size', 'Suppressed  by', 'Suppressing  Agency', 'Facility Identification', 'Other Companies', 'Voltage', 'Equipment Involved With Ignition', 'Type', 'Was There an Outage', 'Date.1', 'Time.1', 'Suspected Initiating Event', 'Equipment /Facility  Failure', 'Contact From  Object', 'Facility  Contacted', 'Contributing Factor', 'Notes', 'Material At Origin', 'HFTD', 'Suppressed By', 'Suppressing Agency', 'Voltage (KVolts)', 'Equipment /Facility Failure', 'Contact From Object', 'Facility Contacted', 'Equipment/Facility Failure', 'HFRA', 'Suppressed by', 'Voltage (Volts)', 'Suspected Ignition Cause', 'Month', 'Day', 'Year', 'Tier', 'Tier 2', 'Tier 3', 'Unnamed: 29', 'FPI#', 'FPI (Level)', 'Structures Destroyed']


c:\Users\Gopal\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Gopal\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [7]:
# Keep essential columns
fire_clean = fire_df[[
    'Latitude',
    'Longitude',
    'Contact From Object',
    'Type',
    'Voltage (KVolts)',
    'HFTD',
    'Contributing Factor'
]].copy()

# Filter vegetation + overhead only
fire_veg = fire_clean[
    fire_clean['Contact From Object'].str.contains('Vegetation', case=False, na=False) &
    (fire_clean['Type'] == 'Overhead')
]

print(f"Total incidents: {len(fire_clean)}")
print(f"Vegetation incidents: {len(fire_veg)}")
print(fire_veg.head())

Total incidents: 1786
Vegetation incidents: 46
       Latitude   Longitude Contact From Object      Type  Voltage (KVolts)  \
1319  34.077055 -117.879650          Vegetation  Overhead              12.0   
1378  35.979526 -116.271327          Vegetation  Overhead               4.0   
1386  34.885290 -117.101510          Vegetation  Overhead              12.0   
1387  34.233211 -116.430735          Vegetation  Overhead              25.0   
1395  33.799264 -117.074382          Vegetation  Overhead              12.0   

             HFTD Contributing Factor  
1319     non-HFTD      External force  
1378     non-HFTD      External Force  
1386     non-HFTD      External Force  
1387     non-HFTD      External Force  
1395  HFTD Tier 2      External Force  


In [8]:
# Drop rows with missing coordinates
fire_veg = fire_veg.dropna(subset=['Latitude', 'Longitude'])

# Convert to GeoDataFrame
fire_gdf = gpd.GeoDataFrame(
    fire_veg,
    geometry=gpd.points_from_xy(fire_veg['Longitude'], fire_veg['Latitude']),
    crs='EPSG:4326'
)

# Match CRS with transmission lines
fire_gdf = fire_gdf.to_crs(lines_filtered.crs)

print(f"Fire GDF rows: {len(fire_gdf)}")
print(f"CRS: {fire_gdf.crs}")
print(fire_gdf.head())

Fire GDF rows: 46
CRS: EPSG:3857
       Latitude   Longitude Contact From Object      Type  Voltage (KVolts)  \
1319  34.077055 -117.879650          Vegetation  Overhead              12.0   
1378  35.979526 -116.271327          Vegetation  Overhead               4.0   
1386  34.885290 -117.101510          Vegetation  Overhead              12.0   
1387  34.233211 -116.430735          Vegetation  Overhead              25.0   
1395  33.799264 -117.074382          Vegetation  Overhead              12.0   

             HFTD Contributing Factor                           geometry  
1319     non-HFTD      External force  POINT (-13122302.613 4039153.329)  
1378     non-HFTD      External Force  POINT (-12943264.915 4297804.547)  
1386     non-HFTD      External Force  POINT (-13035680.464 4148303.419)  
1387     non-HFTD      External Force  POINT (-12961010.133 4060159.717)  
1395  HFTD Tier 2      External Force  POINT (-13032660.589 4001879.829)  


In [9]:
# Create 500m buffer around each transmission line
lines_buffered = lines_filtered.copy()
lines_buffered['geometry'] = lines_filtered.geometry.buffer(500)

# Label segments — 1 if fire incident within buffer, 0 if not
lines_buffered['risk_label'] = 0

for idx, fire_point in fire_gdf.iterrows():
    intersects = lines_buffered.geometry.intersects(fire_point.geometry)
    lines_buffered.loc[intersects, 'risk_label'] = 1

print(f"Total segments: {len(lines_buffered)}")
print(f"High risk segments: {lines_buffered['risk_label'].sum()}")
print(f"Low risk segments: {(lines_buffered['risk_label'] == 0).sum()}")

Total segments: 6675
High risk segments: 30
Low risk segments: 6645


In [10]:
# Extract centroids for GEE sampling
lines_centroids = lines_filtered.copy()
lines_centroids['geometry'] = lines_filtered.geometry.centroid

# Convert to lat/lon for GEE
lines_centroids = lines_centroids.to_crs('EPSG:4326')
lines_centroids['lon'] = lines_centroids.geometry.x
lines_centroids['lat'] = lines_centroids.geometry.y

# Add risk label
lines_centroids['risk_label'] = lines_buffered['risk_label'].values

print(f"Centroids created: {len(lines_centroids)}")
print(lines_centroids[['Name', 'kV', 'lat', 'lon', 'risk_label']].head())

Centroids created: 6675
        Name     kV        lat         lon  risk_label
0  AMP 115kV  115.0  37.765869 -122.228364           0
1  AMP 115kV  115.0  37.776214 -122.268504           0
2  AMP 115kV  115.0  37.784227 -122.291273           0
3  AMP 115kV  115.0  37.791645 -122.281351           0
4  ANZA 34kV   34.0  33.595116 -116.682408           0


In [11]:
# Extract NDVI from Sentinel-2 via GEE
def get_ndvi(lon, lat):
    point = ee.Geometry.Point([lon, lat])
    
    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(point) \
        .filterDate('2023-01-01', '2024-01-01') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
        .median()
    
    ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
    
    result = ndvi.sample(point, 10).first().getInfo()
    
    if result:
        return result['properties']['NDVI']
    return None

# Test on first point
test_ndvi = get_ndvi(
    lines_centroids['lon'].iloc[0],
    lines_centroids['lat'].iloc[0]
)
print(f"Test NDVI value: {test_ndvi}")

Test NDVI value: 0.24100475013256073


In [19]:
import math

# Split centroids into batches of 500
batch_size = 500
total_batches = math.ceil(len(lines_centroids) / batch_size)

print(f"Total batches: {total_batches}")

# Get Sentinel-2 NDVI image
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate('2023-01-01', '2025-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .median()

ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Export each batch separately
tasks = []
for i in range(total_batches):
    batch = lines_centroids.iloc[i*batch_size:(i+1)*batch_size]
    
    features = [
        ee.Feature(ee.Geometry.Point([row['lon'], row['lat']]), {'id': idx})
        for idx, row in batch.iterrows()
    ]
    
    fc = ee.FeatureCollection(features)
    
    sampled = ndvi.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.mean(),
        scale=10
    )
    
    task = ee.batch.Export.table.toDrive(
        collection=sampled,
        description=f'ndvi_batch_{i}',
        fileFormat='CSV',
        folder='CalVegWatch_upd'
    )
    task.start()
    tasks.append(task)
    print(f"Batch {i+1}/{total_batches} started")

print("All batches started!")

Total batches: 14
Batch 1/14 started
Batch 2/14 started
Batch 3/14 started
Batch 4/14 started
Batch 5/14 started
Batch 6/14 started
Batch 7/14 started
Batch 8/14 started
Batch 9/14 started
Batch 10/14 started
Batch 11/14 started
Batch 12/14 started
Batch 13/14 started
Batch 14/14 started
All batches started!


In [26]:
print(task.status()['state'])

COMPLETED


In [25]:
for i, task in enumerate(tasks):
    print(f"Batch {i+1}: {task.status()['state']}")

Batch 1: COMPLETED
Batch 2: COMPLETED
Batch 3: COMPLETED
Batch 4: COMPLETED
Batch 5: COMPLETED
Batch 6: COMPLETED
Batch 7: COMPLETED
Batch 8: COMPLETED
Batch 9: COMPLETED
Batch 10: COMPLETED
Batch 11: COMPLETED
Batch 12: COMPLETED
Batch 13: COMPLETED
Batch 14: COMPLETED


In [27]:
# Load all NDVI batch CSVs
ndvi_path = r"D:\Profession\My_Space\DS\ML\My_Projects\Vegetation_Encroachment_Risk_Detection_California_Transmission_Lines\Data\ndvi_data"

ndvi_files = glob.glob(os.path.join(ndvi_path, "*.csv"))
print(f"Found {len(ndvi_files)} NDVI files")

# Merge all
ndvi_dfs = []
for file in ndvi_files:
    df = pd.read_csv(file)
    ndvi_dfs.append(df)

ndvi_df = pd.concat(ndvi_dfs, ignore_index=True)

print(f"Total NDVI rows: {len(ndvi_df)}")
print(ndvi_df.head())

Found 14 NDVI files
Total NDVI rows: 6675
   system:index  id      mean  \
0             0   0  0.261563   
1             1   1  0.502851   
2             2   2  0.260692   
3             3   3  0.050684   
4             4   4  0.602863   

                                                .geo  
0  {"type":"Point","coordinates":[-122.2283644894...  
1  {"type":"Point","coordinates":[-122.2685041654...  
2  {"type":"Point","coordinates":[-122.2912732132...  
3  {"type":"Point","coordinates":[-122.2813507801...  
4  {"type":"Point","coordinates":[-116.6824082306...  


In [28]:
# Sort by id to ensure correct order
ndvi_df = ndvi_df.sort_values('id').reset_index(drop=True)

# Add NDVI to transmission lines
lines_centroids['NDVI'] = ndvi_df['mean'].values

# Check
print(f"Lines with NDVI: {lines_centroids['NDVI'].notna().sum()}")
print(lines_centroids[['Name', 'kV', 'lat', 'lon', 'NDVI', 'risk_label']].head())

Lines with NDVI: 6675
        Name     kV        lat         lon      NDVI  risk_label
0  AMP 115kV  115.0  37.765869 -122.228364  0.261563           0
1  AMP 115kV  115.0  37.776214 -122.268504  0.502851           0
2  AMP 115kV  115.0  37.784227 -122.291273  0.260692           0
3  AMP 115kV  115.0  37.791645 -122.281351  0.050684           0
4  ANZA 34kV   34.0  33.595116 -116.682408  0.602863           0


In [29]:
# Save master dataset
lines_centroids.to_csv(
    r"D:\Profession\My_Space\DS\ML\My_Projects\Vegetation_Encroachment_Risk_Detection_California_Transmission_Lines\Data\master_dataset.csv",
    index=False
)

print("Master dataset saved!")
print(f"Total rows: {len(lines_centroids)}")
print(f"High risk segments: {lines_centroids['risk_label'].sum()}")
print(f"Low risk segments: {(lines_centroids['risk_label']==0).sum()}")

Master dataset saved!
Total rows: 6675
High risk segments: 30
Low risk segments: 6645


In [30]:
# Extract slope from DEM
dem = ee.Image('USGS/SRTMGL1_003')
slope = ee.Terrain.slope(dem).rename('slope')

# Test on first point
test_slope = slope.sample(
    ee.Geometry.Point([lines_centroids['lon'].iloc[0], 
                       lines_centroids['lat'].iloc[0]]), 10
).first().getInfo()

print(f"Test slope value: {test_slope['properties']['slope']}")

Test slope value: 2.5215542316436768


In [31]:
# Export slope in batches
slope_tasks = []

for i in range(total_batches):
    batch = lines_centroids.iloc[i*batch_size:(i+1)*batch_size]
    
    features = [
        ee.Feature(ee.Geometry.Point([row['lon'], row['lat']]), {'id': idx})
        for idx, row in batch.iterrows()
    ]
    
    fc = ee.FeatureCollection(features)
    
    sampled = slope.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.mean(),
        scale=30
    )
    
    task = ee.batch.Export.table.toDrive(
        collection=sampled,
        description=f'slope_batch_{i}',
        fileFormat='CSV',
        folder='CalVegWatch_slope'
    )
    task.start()
    slope_tasks.append(task)
    print(f"Slope batch {i+1}/{total_batches} started")

print("All slope batches started!")

Slope batch 1/14 started
Slope batch 2/14 started
Slope batch 3/14 started
Slope batch 4/14 started
Slope batch 5/14 started
Slope batch 6/14 started
Slope batch 7/14 started
Slope batch 8/14 started
Slope batch 9/14 started
Slope batch 10/14 started
Slope batch 11/14 started
Slope batch 12/14 started
Slope batch 13/14 started
Slope batch 14/14 started
All slope batches started!


In [32]:
for i, task in enumerate(slope_tasks):
    print(f"Slope Batch {i+1}: {task.status()['state']}")

Slope Batch 1: COMPLETED
Slope Batch 2: COMPLETED
Slope Batch 3: COMPLETED
Slope Batch 4: COMPLETED
Slope Batch 5: COMPLETED
Slope Batch 6: COMPLETED
Slope Batch 7: COMPLETED
Slope Batch 8: COMPLETED
Slope Batch 9: COMPLETED
Slope Batch 10: COMPLETED
Slope Batch 11: COMPLETED
Slope Batch 12: COMPLETED
Slope Batch 13: COMPLETED
Slope Batch 14: COMPLETED


In [33]:
# Load all slope batch CSVs
slope_path = r"D:\Profession\My_Space\DS\ML\My_Projects\Vegetation_Encroachment_Risk_Detection_California_Transmission_Lines\Data\slope_data"

slope_files = glob.glob(os.path.join(slope_path, "*.csv"))
print(f"Found {len(slope_files)} slope files")

# Merge all
slope_dfs = []
for file in slope_files:
    df = pd.read_csv(file)
    slope_dfs.append(df)

slope_df = pd.concat(slope_dfs, ignore_index=True)
slope_df = slope_df.sort_values('id').reset_index(drop=True)

print(f"Total slope rows: {len(slope_df)}")
print(slope_df.head())

Found 14 slope files
Total slope rows: 6675
   system:index  id      mean  \
0             0   0  2.521554   
1             1   1  2.988725   
2             2   2  6.870550   
3             3   3  0.927410   
4             4   4  4.339175   

                                                .geo  
0  {"type":"Point","coordinates":[-122.2283644894...  
1  {"type":"Point","coordinates":[-122.2685041654...  
2  {"type":"Point","coordinates":[-122.2912732132...  
3  {"type":"Point","coordinates":[-122.2813507801...  
4  {"type":"Point","coordinates":[-116.6824082306...  


In [34]:
# Add slope to lines_centroids
lines_centroids['slope'] = slope_df['mean'].values

print(f"Lines with slope: {lines_centroids['slope'].notna().sum()}")
print(lines_centroids[['Name', 'kV', 'NDVI', 'slope', 'risk_label']].head())

Lines with slope: 6675
        Name     kV      NDVI     slope  risk_label
0  AMP 115kV  115.0  0.261563  2.521554           0
1  AMP 115kV  115.0  0.502851  2.988725           0
2  AMP 115kV  115.0  0.260692  6.870550           0
3  AMP 115kV  115.0  0.050684  0.927410           0
4  ANZA 34kV   34.0  0.602863  4.339175           0


In [39]:
datasets = ee.data.listAssets({'parent': 'projects/earthengine-public/assets/USGS/NLCD_RELEASES'})
for d in datasets['assets']:
    print(d['id'])

USGS/NLCD_RELEASES/2016_REL
USGS/NLCD_RELEASES/2019_REL
USGS/NLCD_RELEASES/2020_REL
USGS/NLCD_RELEASES/2021_REL
USGS/NLCD_RELEASES/2023_REL


In [42]:
# Extract Land Cover from NLCD
landcover = ee.ImageCollection('USGS/NLCD_RELEASES/2021_REL/NLCD') \
    .first() \
    .select('landcover')

# Test on first point
test_lc = landcover.sample(
    ee.Geometry.Point([lines_centroids['lon'].iloc[0],
                       lines_centroids['lat'].iloc[0]]), 30
).first().getInfo()

print(f"Test land cover value: {test_lc['properties']['landcover']}")

Test land cover value: 23


In [41]:
# Check what's inside 2021_REL folder
datasets = ee.data.listAssets({'parent': 'projects/earthengine-public/assets/USGS/NLCD_RELEASES/2021_REL'})
for d in datasets['assets']:
    print(d['id'])

USGS/NLCD_RELEASES/2021_REL/NLCD
USGS/NLCD_RELEASES/2021_REL/TCC


In [43]:
# Export land cover in batches
lc_tasks = []

for i in range(total_batches):
    batch = lines_centroids.iloc[i*batch_size:(i+1)*batch_size]
    
    features = [
        ee.Feature(ee.Geometry.Point([row['lon'], row['lat']]), {'id': idx})
        for idx, row in batch.iterrows()
    ]
    
    fc = ee.FeatureCollection(features)
    
    sampled = landcover.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.mode(),
        scale=30
    )
    
    task = ee.batch.Export.table.toDrive(
        collection=sampled,
        description=f'lc_batch_{i}',
        fileFormat='CSV',
        folder='CalVegWatch_lc'
    )
    task.start()
    lc_tasks.append(task)
    print(f"LC batch {i+1}/{total_batches} started")

print("All land cover batches started!")

LC batch 1/14 started
LC batch 2/14 started
LC batch 3/14 started
LC batch 4/14 started
LC batch 5/14 started
LC batch 6/14 started
LC batch 7/14 started
LC batch 8/14 started
LC batch 9/14 started
LC batch 10/14 started
LC batch 11/14 started
LC batch 12/14 started
LC batch 13/14 started
LC batch 14/14 started
All land cover batches started!


In [44]:
for i, task in enumerate(lc_tasks):
    print(f"LC Batch {i+1}: {task.status()['state']}")

LC Batch 1: COMPLETED
LC Batch 2: COMPLETED
LC Batch 3: COMPLETED
LC Batch 4: COMPLETED
LC Batch 5: COMPLETED
LC Batch 6: COMPLETED
LC Batch 7: COMPLETED
LC Batch 8: COMPLETED
LC Batch 9: COMPLETED
LC Batch 10: COMPLETED
LC Batch 11: COMPLETED
LC Batch 12: COMPLETED
LC Batch 13: COMPLETED
LC Batch 14: COMPLETED


In [45]:
# Load all land cover batch CSVs
lc_path = r"D:\Profession\My_Space\DS\ML\My_Projects\Vegetation_Encroachment_Risk_Detection_California_Transmission_Lines\Data\landcover_data"

lc_files = glob.glob(os.path.join(lc_path, "*.csv"))
print(f"Found {len(lc_files)} land cover files")

# Merge all
lc_dfs = []
for file in lc_files:
    df = pd.read_csv(file)
    lc_dfs.append(df)

lc_df = pd.concat(lc_dfs, ignore_index=True)
lc_df = lc_df.sort_values('id').reset_index(drop=True)

print(f"Total land cover rows: {len(lc_df)}")
print(lc_df.head())

Found 14 land cover files
Total land cover rows: 6675
   system:index  id  mode                                               .geo
0             0   0  23.0  {"type":"Point","coordinates":[-122.2283644894...
1             1   1  22.0  {"type":"Point","coordinates":[-122.2685041654...
2             2   2  23.0  {"type":"Point","coordinates":[-122.2912732132...
3             3   3  24.0  {"type":"Point","coordinates":[-122.2813507801...
4             4   4  52.0  {"type":"Point","coordinates":[-116.6824082306...


In [47]:
# Add land cover to lines_centroids
lines_centroids['landcover'] = lc_df['mode'].values

# Save final master dataset
lines_centroids.to_csv(
    r"D:\Profession\My_Space\DS\ML\My_Projects\Vegetation_Encroachment_Risk_Detection_California_Transmission_Lines\Data\master_dataset.csv",
    index=False
)

print("Final master dataset saved!")
print(lines_centroids[['Name', 'kV', 'NDVI', 'slope', 'landcover', 'risk_label']].head())
print(f"\nTotal rows: {len(lines_centroids)}")
print(f"High risk: {lines_centroids['risk_label'].sum()}")
print(f"Low risk: {(lines_centroids['risk_label']==0).sum()}")

Final master dataset saved!
        Name     kV      NDVI     slope  landcover  risk_label
0  AMP 115kV  115.0  0.261563  2.521554       23.0           0
1  AMP 115kV  115.0  0.502851  2.988725       22.0           0
2  AMP 115kV  115.0  0.260692  6.870550       23.0           0
3  AMP 115kV  115.0  0.050684  0.927410       24.0           0
4  ANZA 34kV   34.0  0.602863  4.339175       52.0           0

Total rows: 6675
High risk: 30
Low risk: 6645
